# GPU TMCMC for 5-Species Biofilm (Colab)

JAX GPU-accelerated TMCMC with NUTS/RW mutation, Normalizing Flow proposals, and Waste-free SMC.

**Runtime**: ~2.5 min (200 particles, T4 GPU)

## Setup

In [ ]:
# Cell 1: Install JAX GPU
!pip install -q jax[cuda12] optax
import os
import sys
os.environ['JAX_PLATFORMS'] = 'cuda'
import jax
import jax.numpy as jnp
jax.config.update('jax_enable_x64', True)
print(f'JAX devices: {jax.devices()}')
assert any('cuda' in str(d).lower() or 'gpu' in str(d).lower() for d in jax.devices()), 'No GPU!'

In [ ]:
# Cell 2: Mount Google Drive and extract package
from google.colab import drive
drive.mount('/content/drive')

import zipfile
zip_path = '/content/drive/MyDrive/tmcmc_colab/tmcmc_colab_package.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/tmcmc')

sys.path.insert(0, '/content/tmcmc')
print('Package extracted to /content/tmcmc')
!ls /content/tmcmc/

In [ ]:
# Cell 3: Configuration
# ===== EDIT THESE =====
CONDITION = 'Dysbiotic'        # Commensal / Dysbiotic
CULTIVATION = 'HOBIC'          # Static / HOBIC
N_PARTICLES = 200              # 200 for production, 50 for quick test
MUTATION = 'rw'                # rw / hmc / nuts
N_MUTATION_STEPS = 10          # mutation steps per stage
MAX_STAGES = 30
SEED = 42

# Advanced: Flow-enhanced SMC
USE_FLOW = False               # True to enable NF proposals (CS basin)
FLOW_MIX_RATIO = 0.8          # 80% Flow + 20% RW
FLOW_START_BETA = 0.1

# Advanced: Waste-free SMC
WASTE_FREE = True              # True to enable (recommended!)

# ODE settings
N_STEPS = 2500
DT = 1e-4
K_HILL = 0.05
N_HILL = 4.0
LAMBDA_PG = 5.0
LAMBDA_LATE = 3.0
LAMBDA_RARE = 0.1
START_FROM_DAY = 1
USE_EXP_INIT = True

print(f'Config: {CONDITION} {CULTIVATION}, {N_PARTICLES}p, {MUTATION}, waste_free={WASTE_FREE}, flow={USE_FLOW}')

In [ ]:
# Cell 4: Data loading (self-contained, no pandas dependency chain)
import numpy as np
import pandas as pd
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('colab_tmcmc')

DATA_DIR = Path('/content/tmcmc/experiment_data')
CONFIG_DIR = Path('/content/tmcmc/model_config')

SPECIES_MAP = {
    'Blue': 0, 'Green': 1, 'Yellow': 2, 'Orange': 2, 'Purple': 3, 'Red': 4,
    'S. oralis': 0, 'A. naeslundii': 1, 'V. dispar': 2, 'V. parvula': 2,
    'F. nucleatum': 3, 'P. gingivalis_W83': 4, 'P. gingivalis_20709': 4,
}
SPECIES_MAP_COMMENSAL = {
    'Blue': 0, 'Green': 1, 'Yellow': 2, 'Purple': 3, 'Red': 4,
    'S. oralis': 0, 'A. naeslundii': 1, 'V. dispar': 2,
    'F. nucleatum': 3, 'P. gingivalis_W83': 4, 'P. gingivalis_20709': 4,
}

def load_prior_bounds(condition, cultivation):
    with open(CONFIG_DIR / 'prior_bounds.json') as f:
        config = json.load(f)
    n_base = 20
    bounds = [tuple(config.get('default_bounds', [-1.0, 1.0]))] * n_base
    key = f'{condition}_{cultivation}'
    strategy = config['strategies'].get(key, {
        'locks': [6, 12, 13, 16, 17],
        'bounds': {'3': [0,3], '4': [0,3], '8': [0,3], '9': [0,3], '15': [0,3]}
    })
    locked = []
    for idx in strategy.get('locks', []):
        bounds[idx] = (0.0, 0.0)
        locked.append(idx)
    for idx_str, rng in strategy.get('bounds', {}).items():
        bounds[int(idx_str)] = tuple(rng)
    for idx in locked:
        bounds[idx] = (0.0, 0.0)
    return np.array(bounds[:20], dtype=np.float64)


def load_experimental_data(data_dir, condition, cultivation, start_from_day=1, normalize=True):
    """Load experimental data (self-contained for Colab)."""
    species_map = SPECIES_MAP_COMMENSAL if condition == 'Commensal' else SPECIES_MAP
    
    # Load boxplot (total volume)
    bp_file = data_dir / f'boxplot_{condition}_{cultivation}.csv'
    if not bp_file.exists():
        bp_file = data_dir / 'biofilm_boxplot_data.csv'
    bp_df = pd.read_csv(bp_file)
    if 'condition' in bp_df.columns:
        bp_df = bp_df[(bp_df['condition'] == condition) & (bp_df['cultivation'] == cultivation)]
    
    # Load species distribution
    sp_file = data_dir / 'fig3_species_distribution_summary.csv'
    sp_df = pd.read_csv(sp_file)
    sp_df = sp_df[(sp_df['condition'] == condition) & (sp_df['cultivation'] == cultivation)]
    
    days = sorted(bp_df['day'].unique())
    n_tp = len(days)
    data = np.zeros((n_tp, 5))
    total_vols = np.zeros(n_tp)
    
    for i, day in enumerate(days):
        dv = bp_df[bp_df['day'] == day]
        if len(dv) > 0:
            total_vol = dv['median'].values[0]
            total_vols[i] = total_vol
        for _, row in sp_df[sp_df['day'] == day].iterrows():
            sp = row['species']
            if sp in species_map:
                data[i, species_map[sp]] = total_vol * row['median'] / 100.0
    
    # Per-species sigma from replicates
    rep_file = data_dir / 'fig3_species_distribution_replicates.csv'
    min_sigma = 0.05
    sigma_obs = np.full(5, min_sigma)
    if rep_file.exists():
        rep_df = pd.read_csv(rep_file)
        rep_df = rep_df[(rep_df['condition'] == condition) & (rep_df['cultivation'] == cultivation)]
        rep_map = {'S. oralis': 0, 'A. naeslundii': 1, 'V. dispar': 2, 'V. parvula': 2,
                   'F. nucleatum': 3, 'P. gingivalis_20709': 4, 'P. gingivalis_W83': 4}
        for sp_name, sp_idx in rep_map.items():
            sigs = []
            for day in days:
                vals = rep_df[(rep_df['species'] == sp_name) & (rep_df['day'] == day)]['distribution_pct'].values
                if len(vals) >= 3:
                    iqr = np.percentile(vals, 75) - np.percentile(vals, 25)
                    sigs.append(max(iqr / 1.35 / 100.0, min_sigma))
            if sigs:
                sigma_obs[sp_idx] = max(np.mean(sigs), min_sigma)
    
    data_abs = data.copy()
    if normalize:
        rs = data.sum(axis=1, keepdims=True)
        rs = np.where(rs > 0, rs, 1.0)
        data = data / rs
        # Recompute sigma in fraction space
        if rep_file.exists():
            sigma_frac = np.full(5, min_sigma)
            for sp_name, sp_idx in rep_map.items():
                sp_sigs = []
                for day in days:
                    vals = rep_df[(rep_df['species'] == sp_name) & (rep_df['day'] == day)]['distribution_pct'].values
                    if len(vals) >= 3:
                        iqr = np.percentile(vals, 75) - np.percentile(vals, 25)
                        sp_sigs.append(max(iqr / 1.35 / 100.0, min_sigma))
                if sp_sigs:
                    sigma_frac[sp_idx] = max(np.mean(sp_sigs), min_sigma)
            sigma_obs = sigma_frac
    
    phi_init_exp = data_abs[0, :].copy()
    
    # Filter from start_from_day
    if start_from_day > 1 or USE_EXP_INIT:
        if start_from_day > 1:
            idx = [i for i, d in enumerate(days) if d >= start_from_day]
        else:
            idx = [i for i, d in enumerate(days) if d > 1]
        data = data[idx]
        days = [days[i] for i in idx]
    
    logger.info(f'Data: {data.shape}, days={days}, sigma={sigma_obs}')
    return data, np.array(days, dtype=float), sigma_obs, phi_init_exp


def convert_days_to_model_time(t_days, dt, n_steps):
    t_max = n_steps * dt
    day_scale = (t_max * 0.95) / t_days.max()
    t_model = t_days * day_scale
    idx = np.round(t_model / dt).astype(int)
    idx = np.clip(idx, 0, n_steps)
    return t_model, idx


# Load data
data, t_days, sigma_obs, phi_init_exp = load_experimental_data(
    DATA_DIR, CONDITION, CULTIVATION, START_FROM_DAY)
_, idx_sparse = convert_days_to_model_time(t_days, DT, N_STEPS)

phi_init = phi_init_exp.copy()
if USE_EXP_INIT:
    total = phi_init.sum()
    if total > 0:
        phi_init = phi_init / total
    phi_init = np.clip(phi_init, 0.001, 0.99)
else:
    phi_init = np.full(5, 0.2)

prior_bounds = load_prior_bounds(CONDITION, CULTIVATION)
print(f'idx_sparse: {idx_sparse}')
print(f'phi_init: {phi_init}')
print(f'prior_bounds shape: {prior_bounds.shape}')

In [ ]:
# Cell 5: Build log-likelihood
from hamilton_ode_jax import simulate_0d

obs = jnp.array(data, dtype=jnp.float64)
phi_init_jax = jnp.array(phi_init, dtype=jnp.float64)
idx_jax = jnp.array(idx_sparse, dtype=jnp.int32)
n_obs, n_species = data.shape

# Weights: lambda_pg for Pg, lambda_late for last 2 timepoints
weights = jnp.ones((n_obs, n_species))
weights = weights.at[:, 4].set(LAMBDA_PG)
weights = weights.at[-2:, :].set(weights[-2:, :] * LAMBDA_LATE)
# Downweight rare species
if LAMBDA_RARE < 1.0:
    mean_ab = jnp.mean(obs, axis=0)
    rare_mask = (mean_ab < 0.05).astype(jnp.float64)
    rare_w = rare_mask * LAMBDA_RARE + (1.0 - rare_mask) * 1.0
    weights = weights * rare_w[jnp.newaxis, :]

def log_likelihood(theta):
    phi_traj = simulate_0d(
        theta, n_steps=N_STEPS, dt=DT,
        phi_init=phi_init_jax, K_hill=K_HILL, n_hill=N_HILL)
    phi_pred = phi_traj[idx_jax, :]
    phi_pred = jnp.clip(phi_pred, 1e-10, 1.0 - 1e-10)
    phi_sum = jnp.sum(phi_pred, axis=1, keepdims=True)
    phi_pred = phi_pred / jnp.maximum(phi_sum, 1e-12)
    residual = obs - phi_pred
    r_sq = (residual / sigma_obs) ** 2
    return -0.5 * jnp.sum(weights * r_sq)

# JIT warmup
print('JIT warmup...')
_ = jax.jit(log_likelihood)(jnp.zeros(20, dtype=jnp.float64))
_ = jax.jit(jax.value_and_grad(log_likelihood))(jnp.zeros(20, dtype=jnp.float64))
print('Warmup OK')

In [ ]:
# Cell 6: Run TMCMC!
from tmcmc_nuts_engine import tmcmc_engine
import time

t0 = time.time()
result = tmcmc_engine(
    log_likelihood,
    prior_bounds,
    mutation=MUTATION,
    n_particles=N_PARTICLES,
    max_stages=MAX_STAGES,
    seed=SEED,
    n_mutation_steps=N_MUTATION_STEPS,
    # Flow options
    use_flow=USE_FLOW,
    flow_mix_ratio=FLOW_MIX_RATIO,
    flow_start_beta=FLOW_START_BETA,
    # Waste-free option
    waste_free=WASTE_FREE,
)
elapsed = time.time() - t0

theta_MAP = result['theta_MAP']
log_evidence = result.get('log_evidence', None)
print('\n=== DONE ===')
print(f'Stages: {result["n_stages"]}')
print(f'Time: {elapsed:.1f}s')
print(f'Mean accept: {np.mean(result["accept_rates"]):.3f}')
print(f'Max logL: {result["log_likelihoods"].max():.1f}')
if log_evidence is not None:
    print(f'Log evidence: {log_evidence:.2f}')
print(f'\nMAP theta: {theta_MAP}')

In [ ]:
# Cell 7: Evaluate MAP fit
theta_MAP_jax = jnp.array(theta_MAP, dtype=jnp.float64)
phi_traj_map = simulate_0d(
    theta_MAP_jax, n_steps=N_STEPS, dt=DT,
    phi_init=phi_init_jax, K_hill=K_HILL, n_hill=N_HILL)
phi_pred_map = np.array(phi_traj_map[idx_sparse, :])
phi_pred_map = np.clip(phi_pred_map, 1e-10, 1.0 - 1e-10)
phi_pred_map = phi_pred_map / phi_pred_map.sum(axis=1, keepdims=True)

rmse = np.sqrt(np.mean((data - phi_pred_map) ** 2))
mae = np.mean(np.abs(data - phi_pred_map))
species_names = ['So', 'An', 'Vd', 'Fn', 'Pg']
r2_vals = []
for sp in range(5):
    ss_res = np.sum((data[:, sp] - phi_pred_map[:, sp]) ** 2)
    ss_tot = np.sum((data[:, sp] - np.mean(data[:, sp])) ** 2)
    r2_vals.append(1.0 - ss_res / max(ss_tot, 1e-12))

print(f'MAP RMSE={rmse:.4f}, MAE={mae:.4f}')
print(f'R2 mean={np.mean(r2_vals):.3f}')
for name, r2 in zip(species_names, r2_vals):
    print(f'  {name}: R2={r2:.3f}')

In [ ]:
# Cell 8: Plot results
import matplotlib.pyplot as plt

species_names = ['S. oralis', 'A. naeslundii', 'Veillonella', 'F. nucleatum', 'P. gingivalis']
colors = ['#1f77b4', '#2ca02c', '#ffbf00', '#9467bd', '#d62728']

# Time trajectory for MAP
t_model_full = np.arange(N_STEPS + 1) * DT
phi_traj_np = np.array(phi_traj_map)
phi_traj_norm = phi_traj_np / np.maximum(phi_traj_np.sum(axis=1, keepdims=True), 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: trajectories
ax = axes[0]
for sp in range(5):
    ax.plot(t_model_full, phi_traj_norm[:, sp], color=colors[sp], label=species_names[sp], alpha=0.8)
# Plot data points
t_data = idx_sparse * DT
for sp in range(5):
    ax.scatter(t_data, data[:, sp], color=colors[sp], s=60, zorder=5, edgecolors='k', linewidth=0.5)
ax.set_xlabel('Model time')
ax.set_ylabel('Species fraction')
ax.set_title(f'{CONDITION} {CULTIVATION} — MAP fit')
ax.legend(fontsize=8)
ax.set_ylim(-0.02, 1.02)

# Right: posterior samples scatter (first 2 params)
ax = axes[1]
samples = result['samples']
ax.scatter(samples[:, 0], samples[:, 2], alpha=0.3, s=10, c=result['log_likelihoods'], cmap='viridis')
ax.set_xlabel('theta[0] (a11)')
ax.set_ylabel('theta[2] (a22)')
ax.set_title(f'Posterior ({samples.shape[0]} samples)')
plt.colorbar(ax.collections[0], ax=ax, label='logL')

plt.tight_layout()
plt.savefig('/content/tmcmc_result.png', dpi=150)
plt.show()
print('Saved to /content/tmcmc_result.png')

In [ ]:
# Cell 9: Save results to Drive
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = Path(f'/content/drive/MyDrive/tmcmc_colab/results/{CONDITION}_{CULTIVATION}_{ts}')
out_dir.mkdir(parents=True, exist_ok=True)

np.save(out_dir / 'samples.npy', result['samples'])
np.save(out_dir / 'logL.npy', result['log_likelihoods'])
with open(out_dir / 'theta_MAP.json', 'w') as f:
    json.dump({str(i): float(v) for i, v in enumerate(theta_MAP)}, f, indent=2)
with open(out_dir / 'config.json', 'w') as f:
    json.dump({
        'condition': CONDITION, 'cultivation': CULTIVATION,
        'n_particles': N_PARTICLES, 'mutation': MUTATION,
        'n_mutation_steps': N_MUTATION_STEPS,
        'waste_free': WASTE_FREE, 'use_flow': USE_FLOW,
        'n_stages': result['n_stages'],
        'total_time_s': elapsed,
        'mean_accept': float(np.mean(result['accept_rates'])),
        'max_logL': float(result['log_likelihoods'].max()),
        'log_evidence': float(log_evidence) if log_evidence is not None else None,
        'rmse': float(rmse), 'mae': float(mae),
        'r2_mean': float(np.mean(r2_vals)),
    }, f, indent=2)

# Copy plot
import shutil
shutil.copy('/content/tmcmc_result.png', out_dir / 'tmcmc_result.png')

print(f'Results saved to {out_dir}')

## Batch: Run all 4 conditions

Uncomment and run the cell below to process all conditions sequentially.

In [ ]:
# Cell 10: Batch run (optional)
# CONDITIONS = [
#     ('Dysbiotic', 'HOBIC'),
#     ('Commensal', 'Static'),
#     ('Commensal', 'HOBIC'),
#     ('Dysbiotic', 'Static'),
# ]
# 
# for cond, cult in CONDITIONS:
#     print(f'\n{"="*60}')
#     print(f'Running {cond} {cult}...')
#     print(f'{"="*60}')
#     data_i, t_days_i, sigma_i, phi_init_i = load_experimental_data(DATA_DIR, cond, cult, 1)
#     _, idx_i = convert_days_to_model_time(t_days_i, DT, N_STEPS)
#     phi0 = phi_init_i / max(phi_init_i.sum(), 1e-12)
#     phi0 = np.clip(phi0, 0.001, 0.99)
#     bounds_i = load_prior_bounds(cond, cult)
#     
#     obs_i = jnp.array(data_i, dtype=jnp.float64)
#     phi0_jax = jnp.array(phi0, dtype=jnp.float64)
#     idx_jax_i = jnp.array(idx_i, dtype=jnp.int32)
#     w_i = jnp.ones_like(obs_i)
#     w_i = w_i.at[:, 4].set(5.0)
#     w_i = w_i.at[-2:, :].set(w_i[-2:, :] * 3.0)
#     
#     def make_ll(obs_i, phi0_jax, idx_jax_i, sigma_i, w_i):
#         def ll(theta):
#             traj = simulate_0d(theta, n_steps=N_STEPS, dt=DT, phi_init=phi0_jax, K_hill=K_HILL, n_hill=N_HILL)
#             pred = traj[idx_jax_i, :]
#             pred = jnp.clip(pred, 1e-10, 1.0-1e-10)
#             pred = pred / jnp.maximum(pred.sum(axis=1, keepdims=True), 1e-12)
#             return -0.5 * jnp.sum(w_i * ((obs_i - pred) / sigma_i)**2)
#         return ll
#     
#     ll_i = make_ll(obs_i, phi0_jax, idx_jax_i, sigma_i, w_i)
#     _ = jax.jit(ll_i)(jnp.zeros(20, dtype=jnp.float64))  # warmup
#     
#     result_i = tmcmc_engine(
#         ll_i, bounds_i, mutation='rw', n_particles=200,
#         max_stages=30, seed=42, n_mutation_steps=10,
#         waste_free=True, use_flow=(cond == 'Commensal' and cult == 'Static'))
#     
#     ts = datetime.now().strftime('%Y%m%d_%H%M%S')
#     out = Path(f'/content/drive/MyDrive/tmcmc_colab/results/{cond}_{cult}_{ts}')
#     out.mkdir(parents=True, exist_ok=True)
#     np.save(out / 'samples.npy', result_i['samples'])
#     np.save(out / 'logL.npy', result_i['log_likelihoods'])
#     with open(out / 'theta_MAP.json', 'w') as f:
#         json.dump({str(i): float(v) for i, v in enumerate(result_i['theta_MAP'])}, f, indent=2)
#     print(f'Saved to {out}')